# Local Brain Region Segmentation

Segment brain structures in your histological images using the trained
DINOv2-Large + UperNet model (1,328 brain regions, 79.1% mIoU).

**This notebook runs on your laptop — no GPU or Databricks needed.**

## Before you start

1. Install dependencies: `pip install torch transformers matplotlib Pillow numpy`
2. Download the model (~1.2 GB):
   ```
   databricks fs cp -r dbfs:/FileStore/allen_brain_data/models/final-200ep ./models/dinov2-upernet-final
   ```
3. Place your brain tissue images (`.jpg`, `.png`, `.tif`) in the `images/` directory

In [ ]:
# Cell 1 — Check dependencies and environment

import sys
from pathlib import Path

# Verify imports
try:
    import torch
    import numpy as np
    from PIL import Image
    import matplotlib.pyplot as plt
    from transformers import UperNetForSemanticSegmentation, AutoImageProcessor
    print("All dependencies available.")
except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Run: pip install torch transformers matplotlib Pillow numpy")
    sys.exit(1)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("  Running on CPU (inference will be slower but works fine)")

# Check model exists
MODEL_DIR = Path("../models/dinov2-upernet-final")
if MODEL_DIR.exists():
    print(f"Model found: {MODEL_DIR}")
else:
    print(f"Model NOT found at {MODEL_DIR}")
    print("Download with:")
    print("  databricks fs cp -r dbfs:/FileStore/allen_brain_data/models/final-200ep ./models/dinov2-upernet-final")

In [ ]:
# Cell 2 — Configuration

IMAGES_DIR = Path("../images")
OUTPUT_DIR = Path("../inference_results")
OUTPUT_DIR.mkdir(exist_ok=True)

CROP_SIZE = 518  # DINOv2 native resolution
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

print(f"Input:  {IMAGES_DIR.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")

In [ ]:
# Cell 3 — Load model

print(f"Loading model from {MODEL_DIR}...")
processor = AutoImageProcessor.from_pretrained(str(MODEL_DIR))
model = UperNetForSemanticSegmentation.from_pretrained(str(MODEL_DIR))
model = model.to(device)
model.eval()

num_params = sum(p.numel() for p in model.parameters())
num_labels = model.config.num_labels
print(f"Model loaded: {num_params:,} parameters, {num_labels} output classes")
print(f"Input size: {processor.size}")

In [ ]:
# Cell 4 — Find input images

image_files = sorted(
    p for p in IMAGES_DIR.iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
)

if not image_files:
    print(f"No images found in {IMAGES_DIR.resolve()}")
    print(f"Place .jpg / .png / .tif files there and re-run this cell.")
else:
    print(f"Found {len(image_files)} image(s):")
    for p in image_files:
        img = Image.open(p)
        print(f"  {p.name:40s}  {img.size[0]:>5d} x {img.size[1]:<5d}  {img.mode}")

In [ ]:
# Cell 5 — Run inference on all images (center-crop)
#
# Saves for each image:
#   <name>_mask.png          — Segmentation mask at model resolution (uint16)
#   <name>_mask_resized.png  — Segmentation mask at original image size (uint16)
#   <name>_visualization.png — Side-by-side: input | segmentation | overlay

results = {}  # filename -> (image, prediction, resized_prediction)

for image_path in image_files:
    print(f"\nProcessing: {image_path.name}")

    # Load image
    image = Image.open(image_path).convert("RGB")
    original_size = image.size  # (width, height)

    # Preprocess and run inference
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    prediction = logits.argmax(dim=1)[0].cpu().numpy()

    # Resize prediction to original image size
    pred_pil = Image.fromarray(prediction.astype(np.uint16))
    resized_pred = np.array(pred_pil.resize(original_size, resample=Image.NEAREST))

    results[image_path.name] = (image, prediction, resized_pred)

    # Save outputs
    base = image_path.stem
    Image.fromarray(prediction.astype(np.uint16)).save(OUTPUT_DIR / f"{base}_mask.png")
    Image.fromarray(resized_pred.astype(np.uint16)).save(OUTPUT_DIR / f"{base}_mask_resized.png")

    # Save visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image)
    axes[0].set_title("Input Image", fontsize=14)
    axes[0].axis("off")

    axes[1].imshow(prediction, cmap="nipy_spectral", vmin=0, vmax=num_labels)
    n_classes = len(np.unique(prediction))
    axes[1].set_title(f"Segmentation (518x518)\n{n_classes} classes", fontsize=14)
    axes[1].axis("off")

    axes[2].imshow(resized_pred, cmap="nipy_spectral", vmin=0, vmax=num_labels)
    axes[2].set_title(f"Segmentation ({original_size[0]}x{original_size[1]})", fontsize=14)
    axes[2].axis("off")

    plt.suptitle(image_path.name, fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{base}_visualization.png", dpi=150, bbox_inches="tight")
    plt.close()

    unique_classes = len(np.unique(resized_pred))
    print(f"  Detected {unique_classes} / {num_labels} classes")
    print(f"  Saved: {base}_mask.png, {base}_mask_resized.png, {base}_visualization.png")

print(f"\nAll results saved to: {OUTPUT_DIR.resolve()}")

In [ ]:
# Cell 6 — Display results inline

for name, (image, prediction, resized_pred) in results.items():
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(image)
    axes[0].set_title("Input", fontsize=13)
    axes[0].axis("off")

    axes[1].imshow(prediction, cmap="nipy_spectral", vmin=0, vmax=num_labels)
    axes[1].set_title(f"Prediction ({len(np.unique(prediction))} classes)", fontsize=13)
    axes[1].axis("off")

    axes[2].imshow(resized_pred, cmap="nipy_spectral", vmin=0, vmax=num_labels)
    axes[2].set_title("Resized to original", fontsize=13)
    axes[2].axis("off")

    plt.suptitle(name, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Optional: Sliding Window Inference

Center-crop inference processes only a 518x518 pixel region from the center
of each image. **Sliding window** tiles the entire image with overlapping
518x518 windows and averages predictions, giving full-resolution output.

- Captures structures at image edges that center-crop misses
- Slower: ~15 tiles per image vs 1 tile for center-crop
- On CPU: ~2-8 minutes per image (vs ~10-30 seconds for center-crop)
- Uses more memory (logit accumulation buffer)

Run the cell below to apply sliding window inference to the **first** image.
To process all images with sliding window, use the CLI:
```
python scripts/run_inference.py --image-dir images/ --output inference_results/ --sliding-window
```

In [ ]:
# Cell 8 — Sliding window inference on first image

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
STRIDE = CROP_SIZE // 2  # 259 — 50% overlap


def normalize_tile(tile_rgb):
    """RGB array (H, W, 3) uint8 -> tensor (1, 3, H, W) float32."""
    img = tile_rgb.astype(np.float32) / 255.0
    img = img.transpose(2, 0, 1)  # HWC -> CHW
    for c in range(3):
        img[c] = (img[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
    return torch.from_numpy(img).unsqueeze(0)


def sliding_window_predict(model, image_rgb, num_labels, crop_size, stride, device):
    """Full-resolution tiled prediction with overlapping windows."""
    h, w = image_rgb.shape[:2]
    pad_h = max(0, crop_size - h)
    pad_w = max(0, crop_size - w)
    if pad_h > 0 or pad_w > 0:
        image_rgb = np.pad(image_rgb, ((0, pad_h), (0, pad_w), (0, 0)),
                           mode="constant", constant_values=0)
    ph, pw = image_rgb.shape[:2]

    logit_sum = np.zeros((num_labels, ph, pw), dtype=np.float32)
    count_map = np.zeros((ph, pw), dtype=np.float32)

    y_starts = list(range(0, ph - crop_size + 1, stride))
    x_starts = list(range(0, pw - crop_size + 1, stride))
    if y_starts[-1] + crop_size < ph:
        y_starts.append(ph - crop_size)
    if x_starts[-1] + crop_size < pw:
        x_starts.append(pw - crop_size)

    n_tiles = len(y_starts) * len(x_starts)
    use_cuda = device.type == "cuda"
    model.eval()
    tile_idx = 0
    for y in y_starts:
        for x in x_starts:
            tile = image_rgb[y:y + crop_size, x:x + crop_size]
            pv = normalize_tile(tile).to(device)
            with torch.no_grad():
                if use_cuda:
                    with torch.amp.autocast("cuda", dtype=torch.float16):
                        logits = model(pixel_values=pv).logits
                else:
                    logits = model(pixel_values=pv).logits
            tile_logits = logits.squeeze(0).float().cpu().numpy()
            logit_sum[:, y:y + crop_size, x:x + crop_size] += tile_logits
            count_map[y:y + crop_size, x:x + crop_size] += 1.0
            tile_idx += 1
            if tile_idx % 5 == 0 or tile_idx == n_tiles:
                print(f"  Tile {tile_idx}/{n_tiles}", end="\r")

    print()
    count_map = np.maximum(count_map, 1.0)
    avg_logits = logit_sum / count_map[np.newaxis, :, :]
    return avg_logits.argmax(axis=0).astype(np.int64)[:h, :w]


if image_files:
    target = image_files[0]
    print(f"Sliding window inference on: {target.name}")
    img_pil = Image.open(target).convert("RGB")
    img_np = np.array(img_pil)

    sw_pred = sliding_window_predict(
        model, img_np, num_labels, CROP_SIZE, STRIDE, device,
    )

    # Save
    base = target.stem
    Image.fromarray(sw_pred.astype(np.uint16)).save(
        OUTPUT_DIR / f"{base}_mask_sliding_window.png"
    )

    # Display: center-crop vs sliding window
    cc_pred = results[target.name][1]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img_pil)
    axes[0].set_title("Input", fontsize=13)
    axes[0].axis("off")

    axes[1].imshow(cc_pred, cmap="nipy_spectral", vmin=0, vmax=num_labels)
    axes[1].set_title(f"Center-crop ({len(np.unique(cc_pred))} classes)", fontsize=13)
    axes[1].axis("off")

    axes[2].imshow(sw_pred, cmap="nipy_spectral", vmin=0, vmax=num_labels)
    axes[2].set_title(f"Sliding window ({len(np.unique(sw_pred))} classes)", fontsize=13)
    axes[2].axis("off")

    plt.suptitle(f"{target.name} — Center-Crop vs Sliding Window",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"Saved: {base}_mask_sliding_window.png")
else:
    print("No images to process. Add images to images/ and re-run.")

## Understanding the Output

Each pixel in the segmentation mask contains a **class ID** (0-1327) corresponding
to one of 1,328 brain structures from the Allen Mouse Brain Atlas.

- **Class 0** = Background (outside brain tissue)
- **Classes 1-1327** = Brain structures (e.g., Caudoputamen, Hippocampus CA1, Cerebellum)

The `nipy_spectral` colormap maps each class ID to a unique color for visualization.

Use the cell below to look up what brain region the model predicted at a specific
pixel coordinate in your image.

In [ ]:
# Cell 10 — Look up predicted brain region at a pixel coordinate
#
# Change IMAGE_NAME, ROW, COL to explore different locations.

# --- Edit these ---
IMAGE_NAME = image_files[0].name if image_files else ""
ROW = 250   # y coordinate (pixels from top)
COL = 250   # x coordinate (pixels from left)
# ------------------

if IMAGE_NAME and IMAGE_NAME in results:
    _, prediction, resized_pred = results[IMAGE_NAME]

    # Look up class at model resolution
    class_id = int(prediction[min(ROW, prediction.shape[0]-1),
                              min(COL, prediction.shape[1]-1)])

    # Try to get class name from model config
    id2label = getattr(model.config, "id2label", None)
    if id2label and str(class_id) in id2label:
        class_name = id2label[str(class_id)]
    elif id2label and class_id in id2label:
        class_name = id2label[class_id]
    else:
        class_name = f"(id2label not available — class ID {class_id})"

    print(f"Image:    {IMAGE_NAME}")
    print(f"Position: row={ROW}, col={COL}")
    print(f"Class ID: {class_id}")
    print(f"Region:   {class_name}")

    # Show location on prediction
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(prediction, cmap="nipy_spectral", vmin=0, vmax=num_labels)
    ax.plot(COL, ROW, "r+", markersize=20, markeredgewidth=3)
    ax.set_title(f"{IMAGE_NAME} — Class {class_id}: {class_name}",
                 fontsize=12, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No results yet. Run the inference cells above first.")

## Summary

**Output directory:** `inference_results/`

**Files per image:**
- `<name>_mask.png` — Raw segmentation at model resolution (518x518, uint16)
- `<name>_mask_resized.png` — Segmentation at original image size (uint16)
- `<name>_visualization.png` — Side-by-side comparison
- `<name>_mask_sliding_window.png` — Full-resolution tiled prediction (if sliding window was run)

**Batch processing:** For many images, the CLI script is faster:
```bash
python scripts/run_inference.py --image-dir images/ --output inference_results/
```

**More info:**
- [Model download guide](../docs/model_download_guide.md)
- [Paper draft](../docs/paper_draft.md) — full ablation study
- [CLI inference script](../scripts/run_inference.py) — `--help` for all options